# Image Segmentation With PyTorch Lightning

In [ ]:
!pip install -q lightning torchmetrics timm tifffile segmentation-models-pytorch

In [ ]:
import random

import lightning
import matplotlib
import matplotlib.pyplot as plt
import numpy
import pandas
import torch
import torchmetrics
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger
from lightning.pytorch.utilities.model_summary import ModelSummary
from tifffile import imread
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms.v2 import functional as transforms

In [ ]:
!nvidia-smi

In [ ]:
%%time
!rm -rf sample_data
!git clone https://github.com/mlambda/ign-FNF-2009.git

## Loading the Data

The data is made of 20 large map tiles from IGN (the French national mapping agency). Each large tile is itself a square of 3 × 3 tiles of 1024 × 1024 pixels.

2 tiles were set aside for validation, and 2 other tiles for testing.

In [ ]:
df_train = pandas.read_csv("ign-FNF-2009/train-files.csv")
df_val = pandas.read_csv("ign-FNF-2009/val-files.csv")
df_test = pandas.read_csv("ign-FNF-2009/test-files.csv")

print("Train coordinates")
print(df_train.coord.unique())

print("Val coordinates")
print(df_val.coord.unique())

print("Test coordinates")
print(df_test.coord.unique())

df_test

## Studying an Example and Its Annotation

In [ ]:
def read_observation(path: str) -> torch.Tensor:
  """Read a tile as a (channels, height, width) tensor."""
  return torch.from_numpy(imread(path)).permute(2, 0, 1)


def read_annotation(path: str) -> torch.Tensor:
  """Read a one-hot annotation as a (1, height, width) mask."""
  return torch.from_numpy(imread(path)).argmax(dim=-1)[None].float()


example_input = read_observation(df_train.Observations[0])
example_annotation = read_annotation(df_train.Annotations[0])

In [ ]:
# Hack needed for imshow to properly display tiles with only
# forest (otherwise the 1s representing forest pixels are replaced with
# 0s if the tile is made only of 1s).
class NoopNormalize(matplotlib.colors.Normalize):
  def __init__(self, vmin=None, vmax=None, vcenter=None, clip=False):
    super().__init__(vmin, vmax, clip)
    self.vcenter = vcenter

  def __call__(self, value, clip=None):
    return value


_noop_normalize = NoopNormalize()


def to_displayable(image: torch.Tensor) -> numpy.ndarray:
  """Turn a (channels, height, width) tensor into an array imshow can plot."""
  image = image.detach().cpu()
  if image.ndim == 2:
    return image.numpy()
  if image.shape[0] == 3:
    return image.permute(1, 2, 0).numpy()
  if image.shape[0] == 2:
    return image[1].numpy()
  return image[0].numpy()


def display_sample(*images: torch.Tensor) -> None:
  """Show side-by-side an input image, the ground truth and the prediction."""
  figure, axes = plt.subplots(1, len(images), figsize=(18, 18))

  titles = ["Input", "Target", "Prediction"]
  for image, ax, title in zip(images, axes, titles):
    ax.set_title(title)
    ax.imshow(to_displayable(image), norm=_noop_normalize)
    ax.axis("off")
  plt.show()


display_sample(example_input, example_annotation)

## General Options

In [ ]:
# Size of the original images
dim_original = 1024

# Size of the network's input images
dim_input = 512

# Number of channels of the input images
num_channel = 3

# Number of processes used to load the data
num_workers = 2

## Data Augmentation

In image processing, it's extremely important to augment the data through transformations, to make the trained network less sensitive to rotations, crops, etc. Implement the `transform_data` function, which, if `eval` is `True`, will just center-crop the `X` and `Y` images, and otherwise will apply all the training transformations you deem relevant to these same tensors (random crop, flip, rotation, color and brightness noise, etc).

To do so, use the functions of the [`torchvision.transforms.v2.functional`](https://docs.pytorch.org/vision/stable/transforms.html#functional-transforms) module, imported here as `transforms`.

In [ ]:
def transform_data(
  X: torch.Tensor,
  Y: torch.Tensor,
  eval: bool = False,
  cropped_size: tuple[int, int] = (dim_input, dim_input),
) -> tuple[torch.Tensor, torch.Tensor]:
  # Your code here
  return X, Y


# display_sample(*transform_data(example_input, example_annotation))

### Solution

In [ ]:
def transform_data(
  X: torch.Tensor,
  Y: torch.Tensor,
  eval: bool = False,
  cropped_size: tuple[int, int] = (dim_input, dim_input),
) -> tuple[torch.Tensor, torch.Tensor]:
  height, width = X.shape[-2:]
  cropped_height, cropped_width = cropped_size
  if eval:
    # Center crop
    crop_params = dict(
      top=(height - cropped_height) // 2,
      left=(width - cropped_width) // 2,
      height=cropped_height,
      width=cropped_width,
    )
    X = transforms.crop(X, **crop_params)
    Y = transforms.crop(Y, **crop_params)
  else:
    # Random rotation
    k = random.choice([None, 1, 3])
    if k is not None:
      X = torch.rot90(X, k, dims=(-2, -1))
      Y = torch.rot90(Y, k, dims=(-2, -1))

    # Random crop
    crop_params = dict(
      top=random.randint(0, height - cropped_height),
      left=random.randint(0, width - cropped_width),
      height=cropped_height,
      width=cropped_width,
    )
    X = transforms.crop(X, **crop_params)
    Y = transforms.crop(Y, **crop_params)

    # Horizontal flip
    if random.random() > 0.5:
      X = transforms.horizontal_flip(X)
      Y = transforms.horizontal_flip(Y)

    # Vertical flip
    if random.random() > 0.5:
      X = transforms.vertical_flip(X)
      Y = transforms.vertical_flip(Y)
  return X, Y


for _ in range(10):
  display_sample(*transform_data(example_input, example_annotation))

## Creating a PyTorch `Dataset`

To describe our data, we'll use the [`torch.utils.data.Dataset`](https://docs.pytorch.org/docs/stable/data.html#map-style-datasets) class. It mainly needs two methods: `__getitem__`, which is called to get **one** example from its index, and `__len__`, which is called to know the total number of examples. The [`DataLoader`](https://docs.pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader) then groups these examples into batches, shuffles the dataset before each epoch and parallelizes the loading.

In [ ]:
class SegmentationDataset(Dataset):
  """Load samples with appropriate transforms."""

  def __init__(
    self,
    df: pandas.DataFrame,
    eval: bool,
    cropped_size: tuple[int, int] = (dim_input, dim_input),
  ) -> None:
    """Initialize the dataset."""
    # Your code here

  def __len__(self):
    """Compute the number of samples."""
    # Your code here

  def __getitem__(self, index):
    """Generate one sample of data."""
    # Your code here

### Solution

In [ ]:
class SegmentationDataset(Dataset):
  """Load samples for PyTorch."""

  def __init__(
    self,
    df: pandas.DataFrame,
    eval: bool,
    cropped_size: tuple[int, int] = (dim_input, dim_input),
  ) -> None:
    """Initialize the dataset."""
    self.X_paths = df.Observations.to_numpy()
    self.Y_paths = df.Annotations.to_numpy()
    assert self.X_paths.shape[0] == self.Y_paths.shape[0]
    self.eval = eval
    self.cropped_size = cropped_size

  def __len__(self) -> int:
    """Compute the number of samples."""
    return self.X_paths.shape[0]

  def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
    """Generate one sample of data."""
    X, Y = transform_data(
      read_observation(self.X_paths[index]),
      read_annotation(self.Y_paths[index]),
      self.eval,
      self.cropped_size,
    )
    return X.float() / 255.0, Y


def make_loader(df: pandas.DataFrame, batch_size: int, eval: bool) -> DataLoader:
  """Wrap a dataset in a data loader: batching, shuffling and prefetching."""
  return DataLoader(
    SegmentationDataset(df, eval),
    batch_size=batch_size,
    shuffle=not eval,
    num_workers=num_workers,
    persistent_workers=num_workers > 0,
  )


loader = make_loader(df_train, 3, eval=True)
X, Y = next(iter(loader))
X.shape, Y.shape

## Creating the Model

For the model, we need to go from an image of dimension (batch × 3 × height × width) to an output of dimension (batch × 1 × height × width).

Take inspiration from the UNet architecture shown in the slides: an encoder that halves the resolution at each level, a decoder that restores it, and *skip connections* that pass to each level of the decoder the features of the encoder at the same resolution.

You can also read [the paper that introduced U-Net](https://arxiv.org/abs/1505.04597).

In [ ]:
def get_model(starting_depth: int = 32, steps: int = 5) -> nn.Module:
  pass  # Your code here

### Solution

In [ ]:
# Implementation of the U-net model described in the paper
# https://arxiv.org/abs/1505.04597 (see also the slides on UNet)


def downsample() -> nn.MaxPool2d:
  return nn.MaxPool2d(kernel_size=2)


def upsample() -> nn.Upsample:
  return nn.Upsample(scale_factor=2)


def semiblock(in_channels: int, feature_maps: int) -> nn.Sequential:
  return nn.Sequential(
    nn.Conv2d(in_channels, feature_maps, 3, padding="same"),
    nn.BatchNorm2d(feature_maps),
    nn.ReLU(),
    nn.Dropout(0.1),
  )


def block(in_channels: int, feature_maps: int) -> nn.Sequential:
  return nn.Sequential(
    semiblock(in_channels, feature_maps), semiblock(feature_maps, feature_maps)
  )


class UNet(nn.Module):
  """U-net: an encoder, a decoder, and skip connections between the two."""

  def __init__(self, starting_depth: int = 32, steps: int = 5) -> None:
    super().__init__()
    depths = [min(512, starting_depth * 2**i) for i in range(steps)]

    self.downsample = downsample()
    self.upsample = upsample()

    # Encoder: one block per level, the resolution being halved at
    # each level
    self.encoder = nn.ModuleList(
      [
        block(depths[i - 1] if i else num_channel, depth)
        for i, depth in enumerate(depths)
      ]
    )

    # Decoder: at each level, the encoding of the same resolution is concatenated to
    # the upsampled tensor, hence the number of input channels
    self.decoder = nn.ModuleList(
      [
        block(depth + previous_depth, depth)
        for depth, previous_depth in zip(depths[-2::-1], depths[::-1])
      ]
    )

    self.head = nn.Conv2d(depths[0], 1, 1)

  def forward(self, inputs: torch.Tensor) -> torch.Tensor:
    encodings = []
    for i, encoder_block in enumerate(self.encoder):
      encodings.append(encoder_block(self.downsample(encodings[-1]) if i else inputs))

    x = encodings[-1]
    for decoder_block, encoding in zip(self.decoder, encodings[-2::-1]):
      x = decoder_block(torch.cat([encoding, self.upsample(x)], dim=1))

    return self.head(x)


def get_model(starting_depth: int = 32, steps: int = 5) -> nn.Module:
  return UNet(starting_depth, steps)

## Training

We can now use our `Dataset` to train our model. The `LightningModule` below describes the loss function, the metrics and the optimizer: the model returns logits, the sigmoid being applied by the loss function during training and explicitly at prediction time.

We track the pixel accuracy and the IoU (see the "Segmentation Losses and Metrics" slide): with imbalanced classes, the accuracy can be high even when the model misses the minority class.

In [ ]:
class ImageSegmenter(lightning.LightningModule):
  """Lightning wrapper for binary segmentation models."""

  def __init__(self, model: nn.Module, learning_rate: float = 1e-4) -> None:
    super().__init__()
    self.save_hyperparameters(ignore=["model"])
    self.model = model
    # An example input lets Lightning display the shapes of the input and output
    # tensors of each layer in the model summary
    self.example_input_array = torch.zeros(1, num_channel, dim_input, dim_input)
    # torchmetrics needs one metric instance per stage
    self.accuracies = nn.ModuleDict(
      {
        f"{stage}_accuracy": torchmetrics.Accuracy(task="binary")
        for stage in ("train", "val", "test")
      }
    )
    # IoU (Jaccard index) of the positive class: unlike accuracy, it isn't
    # inflated by the pixels of a majority class
    self.ious = nn.ModuleDict(
      {
        f"{stage}_iou": torchmetrics.classification.BinaryJaccardIndex()
        for stage in ("train", "val", "test")
      }
    )

  def forward(self, images: torch.Tensor) -> torch.Tensor:
    return self.model(images)

  def _step(self, batch: tuple[torch.Tensor, torch.Tensor], stage: str) -> torch.Tensor:
    images, annotations = batch
    logits = self(images)
    loss = nn.functional.binary_cross_entropy_with_logits(logits, annotations)
    accuracy = self.accuracies[f"{stage}_accuracy"]
    accuracy(logits.sigmoid(), annotations.int())
    self.log(f"{stage}_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
    self.log(f"{stage}_accuracy", accuracy, on_step=False, on_epoch=True, prog_bar=True)
    iou = self.ious[f"{stage}_iou"]
    iou(logits.sigmoid(), annotations.int())
    self.log(f"{stage}_iou", iou, on_step=False, on_epoch=True, prog_bar=True)
    return loss

  def training_step(
    self, batch: tuple[torch.Tensor, torch.Tensor], batch_index: int
  ) -> torch.Tensor:
    return self._step(batch, "train")

  def validation_step(
    self, batch: tuple[torch.Tensor, torch.Tensor], batch_index: int
  ) -> torch.Tensor:
    return self._step(batch, "val")

  def test_step(
    self, batch: tuple[torch.Tensor, torch.Tensor], batch_index: int
  ) -> torch.Tensor:
    return self._step(batch, "test")

  def predict_step(
    self, batch: torch.Tensor | tuple[torch.Tensor, ...], batch_index: int
  ) -> torch.Tensor:
    images = batch[0] if isinstance(batch, (list, tuple)) else batch
    return self(images).sigmoid().cpu()

  def configure_optimizers(self) -> torch.optim.Optimizer:
    return torch.optim.Adam(self.parameters(), lr=self.hparams.learning_rate)


# Test results of each model, compared at the end of the notebook
test_results: dict[str, dict[str, float]] = {}

In [ ]:
# Your code here

### Solution

In [ ]:
# Build the model
segmenter = ImageSegmenter(get_model(), learning_rate=1e-4)
print(ModelSummary(segmenter, max_depth=-1))

batch_size = 4

# Create the data loaders
train_loader = make_loader(df_train, batch_size, eval=False)
val_loader = make_loader(df_val, batch_size, eval=True)

# Train the model, evaluating on the validation loader at the end of
# each epoch
epochs = 15
trainer = lightning.Trainer(
  max_epochs=epochs,
  accelerator="auto",
  devices=1,
  logger=CSVLogger("logs", name="u_net"),
  callbacks=[
    EarlyStopping(monitor="val_loss", patience=5),
    ModelCheckpoint(
      monitor="val_iou", mode="max", save_weights_only=True, save_top_k=1
    ),
  ],
)
trainer.fit(segmenter, train_loader, val_loader)

## Applying the Model to the Test Data

Once your model is trained enough, apply it to the test data: are the predictions of good quality?

In [ ]:
# Your code here

### Solution

In [ ]:
test_loader = make_loader(df_test, 8, eval=True)

# `ckpt_path="best"` reloads the weights saved by the ModelCheckpoint
test_results["UNet"] = trainer.test(
  segmenter, dataloaders=test_loader, ckpt_path="best"
)[0]

# `predict` returns a list of tensors, one per batch: concatenate them
preds = torch.cat(trainer.predict(segmenter, test_loader))

print(len(test_loader.dataset), preds.shape)

for i, p in enumerate(preds[:8]):
  X, Y = test_loader.dataset[i]
  display_sample(X, Y, p)

## Pre-trained Model Without Skip Connections

We now replace the encoder with a network pre-trained on ImageNet, EfficientNet-B7, frozen. The decoder asked for here is deliberately simple: it only upsamples the encoder's output (16x16 maps), **without skip connections**. This model shows what is lost without them: the encoder is excellent, but the fine details of the first layers never reach the decoder.

Complete the model below, train it, then compare its predictions with those of the UNet trained from scratch.

In [ ]:
import timm


class Normalize(nn.Module):
  """Normalize images already scaled to [0, 1], channel by channel."""

  def __init__(
    self, mean: tuple[float, ...] | float, std: tuple[float, ...] | float
  ) -> None:
    super().__init__()
    self.register_buffer("mean", torch.tensor(mean).reshape(1, -1, 1, 1))
    self.register_buffer("std", torch.tensor(std).reshape(1, -1, 1, 1))

  def forward(self, images: torch.Tensor) -> torch.Tensor:
    return (images - self.mean) / self.std


class Frozen(nn.Module):
  """Wrap a module whose weights are frozen and kept in evaluation mode."""

  def __init__(self, module: nn.Module) -> None:
    super().__init__()
    self.module = module.requires_grad_(False)
    self.eval()

  def train(self, mode: bool = True) -> "Frozen":
    return super().train(False)

  def forward(self, images: torch.Tensor) -> torch.Tensor:
    return self.module(images)


base_model = timm.create_model(
  "tf_efficientnet_b7.aa_in1k", pretrained=True, num_classes=0, global_pool=""
)
data_config = timm.data.resolve_model_data_config(base_model)

model = nn.Sequential(
  Normalize(data_config["mean"], data_config["std"]),
  # (2560, 16, 16)
  Frozen(base_model),
  # Your code
)

segmenter = ImageSegmenter(model, learning_rate=1e-4)

print(ModelSummary(segmenter, max_depth=2))

# trainer = lightning.Trainer(
#     max_epochs=epochs,
#     accelerator="auto",
#     devices=1,
#     logger=CSVLogger("logs", name="efficientnet_b7"),
#     callbacks=[EarlyStopping(monitor="val_loss", patience=5),
#                ModelCheckpoint(monitor="val_iou",
#                                mode="max",
#                                save_weights_only=True,
#                                save_top_k=1)])
# trainer.fit(segmenter, train_loader, val_loader)

### Solution

In [ ]:
base_model = timm.create_model(
  "tf_efficientnet_b7.aa_in1k", pretrained=True, num_classes=0, global_pool=""
)
data_config = timm.data.resolve_model_data_config(base_model)

starting_depth = 512


def conv(in_channels: int, out_channels: int, kernel_size: int = 3) -> nn.Sequential:
  return nn.Sequential(
    nn.Conv2d(in_channels, out_channels, kernel_size, padding="same"), nn.ReLU()
  )


model = nn.Sequential(
  Normalize(data_config["mean"], data_config["std"]),
  # (2560, 16, 16)
  Frozen(base_model),
  # (512, 16, 16)
  conv(base_model.num_features, starting_depth, kernel_size=1),
  # (512, 32, 32)
  nn.Upsample(scale_factor=2),
  # (256, 32, 32)
  conv(starting_depth, starting_depth // 2),
  # (256, 64, 64)
  nn.Upsample(scale_factor=2),
  # (128, 64, 64)
  conv(starting_depth // 2, starting_depth // 4),
  # (128, 128, 128)
  nn.Upsample(scale_factor=2),
  # (64, 128, 128)
  conv(starting_depth // 4, starting_depth // 8),
  # (64, 256, 256)
  nn.Upsample(scale_factor=2),
  # (32, 256, 256)
  conv(starting_depth // 8, starting_depth // 16),
  # (32, 512, 512)
  nn.Upsample(scale_factor=2),
  # (16, 512, 512)
  conv(starting_depth // 16, starting_depth // 32),
  nn.Dropout(0.1),
  # (1, 512, 512)
  nn.Conv2d(starting_depth // 32, 1, 3, padding="same"),
)

segmenter = ImageSegmenter(model, learning_rate=1e-4)

print(ModelSummary(segmenter, max_depth=2))

trainer = lightning.Trainer(
  max_epochs=epochs,
  accelerator="auto",
  devices=1,
  logger=CSVLogger("logs", name="efficientnet_b7"),
  callbacks=[
    EarlyStopping(monitor="val_loss", patience=5),
    ModelCheckpoint(
      monitor="val_iou", mode="max", save_weights_only=True, save_top_k=1
    ),
  ],
)
trainer.fit(segmenter, train_loader, val_loader)

In [ ]:
test_loader = make_loader(df_test, 8, eval=True)

# `ckpt_path="best"` reloads the weights saved by the ModelCheckpoint
test_results["EfficientNet-B7 without skip connections"] = trainer.test(
  segmenter, dataloaders=test_loader, ckpt_path="best"
)[0]

# `predict` returns a list of tensors, one per batch: concatenate them
preds = torch.cat(trainer.predict(segmenter, test_loader))

print(len(test_loader.dataset), preds.shape)

for i, p in enumerate(preds[:8]):
  X, Y = test_loader.dataset[i]
  display_sample(X, Y, p)

The masks are much coarser than the UNet's: the decoder rebuilds the contours from 16x16 maps only. The pre-trained encoder recognizes the forest, but without skip connections, the decoder no longer knows precisely *where* its edges are.

## Pre-trained UNet

Let's combine both: a real UNet, with skip connections, whose encoder is the same pre-trained EfficientNet-B7 (see the "UNet with a Pretrained Encoder" slide). The [`segmentation_models_pytorch`](https://smp.readthedocs.io/) (smp) library builds it: `smp.Unet(encoder_name="tu-tf_efficientnet_b7", encoder_weights="imagenet", classes=1)` (the `tu-` prefix designates a `timm` encoder). Only the skip connections set it apart from the previous model.

1. Normalize the inputs as before (`Normalize` with `data_config`), placed before the UNet.
2. First freeze the encoder, `unet.encoder = Frozen(unet.encoder)`, and train the decoder alone.
3. Evaluate the model on the test set and compare its predictions with the two previous models.

In [ ]:
# Your code here

### Solution

In [ ]:
import segmentation_models_pytorch as smp

unet = smp.Unet(
  encoder_name="tu-tf_efficientnet_b7", encoder_weights="imagenet", classes=1
)
# Stage 1: the encoder is frozen (weights and BatchNorm statistics), only the
# decoder learns
unet.encoder = Frozen(unet.encoder)
pretrained_unet = nn.Sequential(Normalize(data_config["mean"], data_config["std"]), unet)

segmenter = ImageSegmenter(pretrained_unet, learning_rate=1e-4)
print(ModelSummary(segmenter, max_depth=2))

trainer = lightning.Trainer(
  max_epochs=epochs,
  accelerator="auto",
  devices=1,
  logger=CSVLogger("logs", name="pretrained_unet"),
  callbacks=[
    EarlyStopping(monitor="val_loss", patience=5),
    ModelCheckpoint(monitor="val_iou", mode="max", save_weights_only=True, save_top_k=1),
  ],
)
trainer.fit(segmenter, train_loader, val_loader)

In [ ]:
# `ckpt_path="best"` also reloads the best weights into `segmenter`, where
# fine-tuning will start from
test_results["Pre-trained UNet, frozen encoder"] = trainer.test(
  segmenter, dataloaders=test_loader, ckpt_path="best"
)[0]
preds = torch.cat(trainer.predict(segmenter, test_loader))
for i, p in enumerate(preds[:8]):
  X, Y = test_loader.dataset[i]
  display_sample(X, Y, p)

### Fine-Tuning the Encoder

Once the decoder is trained, unfreeze the encoder (`unet.encoder = unet.encoder.module`, then `requires_grad_(True)`) and keep training for a few epochs, with AdamW and two parameter groups: a learning rate of 1e-5 for the encoder, whose weights are already good, and 1e-4 for the rest. The whole network being trained, reduce the batch size if GPU memory runs short.

In [ ]:
# Your code here

### Solution

In [ ]:
class FineTuningSegmenter(ImageSegmenter):
  """ImageSegmenter whose encoder has its own, smaller, learning rate."""

  def __init__(
    self, model: nn.Module, encoder_learning_rate: float = 1e-5, **kwargs
  ) -> None:
    super().__init__(model, **kwargs)
    self.encoder_learning_rate = encoder_learning_rate

  def configure_optimizers(self) -> torch.optim.Optimizer:
    parameters = list(self.named_parameters())
    encoder = [p for name, p in parameters if ".encoder." in name]
    others = [p for name, p in parameters if ".encoder." not in name]
    return torch.optim.AdamW(
      [
        {"params": encoder, "lr": self.encoder_learning_rate},
        {"params": others, "lr": self.hparams.learning_rate},
      ]
    )


# Stage 2: the encoder gets back trainable weights and its train() mode
unet.encoder = unet.encoder.module.requires_grad_(True)
segmenter = FineTuningSegmenter(pretrained_unet, encoder_learning_rate=1e-5, learning_rate=1e-4)

# The whole network is trained: smaller batches to fit in GPU memory
fine_tuning_loader = make_loader(df_train, 2, eval=False)
trainer = lightning.Trainer(
  max_epochs=5,
  accelerator="auto",
  devices=1,
  logger=CSVLogger("logs", name="fine_tuned_unet"),
  callbacks=[
    ModelCheckpoint(monitor="val_iou", mode="max", save_weights_only=True, save_top_k=1)
  ],
)
trainer.fit(segmenter, fine_tuning_loader, val_loader)
test_results["Pre-trained UNet, fine-tuned"] = trainer.test(
  segmenter, dataloaders=test_loader, ckpt_path="best"
)[0]

## Comparing the Models

The next cell compares the test results of every trained model. Which model gets the best IoU? Why are the accuracy gaps smaller than the IoU gaps?

In [ ]:
pandas.DataFrame(test_results).T[["test_iou", "test_accuracy", "test_loss"]]

### Solution

Usually, the pre-trained UNet does best, especially once fine-tuned: it combines an encoder that learned to recognize textures on ImageNet with skip connections that keep the contours precise. The model without skip connections is penalized on the edges, the UNet trained from scratch by the small amount of data.

Accuracy counts every pixel, the vast majority of which are easy (the inside of forest or non-forest areas): the models only differ on a small fraction of pixels, near the edges. The IoU only looks at the positive class, and penalizes every missed or extra pixel: it separates them much better.